# 03 - Build Processed Dataset

Este notebook LE o CSV bruto e ESCREVE o dataset analitico (`data/processed/loans_clean.parquet`).
Implementa `docs/cleaning_decisions.md` linha por linha. Se algo aqui divergir do documento,
o notebook para (assert) e sinaliza — nao decide sozinho.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

RAW_PATH = Path('..') / 'data' / 'raw' / 'accepted_2007_to_2018Q4.csv'
PROCESSED_DIR = Path('..') / 'data' / 'processed'
CHUNK_SIZE = 50000
CUTOFF_36 = pd.Timestamp('2015-12-01')
CUTOFF_60 = pd.Timestamp('2013-12-01')


## 1. Populacao aprovada

loan_status em {Fully Paid, Charged Off}; term 36m com issue_d ate dez/2015; term 60m ate
dez/2013. N esperado = 673.553. Se nao bater, o assert interrompe o notebook.

In [2]:
pop_chunks = []
n_total_file = 0
for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    n_total_file += len(chunk)
    status_ok = chunk['loan_status'].isin(['Fully Paid', 'Charged Off'])
    issue_dt = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
    term_clean = chunk['term'].str.strip()
    is_36 = term_clean == '36 months'
    is_60 = term_clean == '60 months'
    mature = (is_36 & issue_dt.notna() & (issue_dt <= CUTOFF_36)) | (is_60 & issue_dt.notna() & (issue_dt <= CUTOFF_60))
    pop_mask = status_ok & mature
    if pop_mask.any():
        pop_chunks.append(chunk.loc[pop_mask].copy())

df = pd.concat(pop_chunks, ignore_index=True)
del pop_chunks

print(f'Linhas lidas do arquivo inteiro: {n_total_file:,}')
print(f'N da populacao: {len(df):,}')
assert len(df) == 673553, f'N inesperado: {len(df)} (esperado 673553) - PARANDO, diverge do contrato.'
print('N confere com o esperado (673.553). Prosseguindo.')


Linhas lidas do arquivo inteiro: 2,260,701
N da populacao: 673,553
N confere com o esperado (673.553). Prosseguindo.


## 2. Alvo

In [3]:
df['target'] = df['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})
print('target criado. Distribuicao:')
print(df['target'].value_counts())
print(f'Taxa de default: {df["target"].mean() * 100:.4f}%')


target criado. Distribuicao:
target
0    573768
1     99785
Name: count, dtype: int64
Taxa de default: 14.8147%


## 3. Remocao das 5 linhas com dti > 100 (erro, nao outlier legitimo)

In [4]:
n_before = len(df)
df = df.loc[~(df['dti'] > 100)].reset_index(drop=True)
n_removed = n_before - len(df)
print(f'Linhas removidas (dti > 100): {n_removed} (esperado: 5)')
assert n_removed == 5, f'Esperava remover 5 linhas, removeu {n_removed} - PARANDO, diverge do contrato.'
print(f'Novo N: {len(df):,}')


Linhas removidas (dti > 100): 5 (esperado: 5)
Novo N: 673,548


## Transformacoes mecanicas (documentadas em cleaning_decisions.md)

Datas para datetime real; `term` para meses (numero, mantendo o nome — e EVAL_ONLY e precisa
ser numerico para o calculo de resultado financeiro); `emp_length` parseado para
`emp_length_anos` (coluna nova, a string original e descartada por redundancia); categoricas
de baixa cardinalidade com strip+lower (cardinalidade nao muda, ja documentado).

In [5]:
date_cols = ['issue_d', 'earliest_cr_line']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format='%b-%Y', errors='coerce')
print('Datas convertidas:', date_cols)

df['term'] = df['term'].str.strip().str.extract(r'(\d+)').astype(float)
print('term convertido para numerico (meses). Valores unicos:', sorted(df['term'].dropna().unique()))

def parse_emp_length(v):
    if pd.isna(v):
        return np.nan
    v = v.strip()
    if v == '< 1 year':
        return 0.0
    if v == '10+ years':
        return 10.0
    digits = ''.join(ch for ch in v if ch.isdigit())
    return float(digits) if digits else np.nan

df['emp_length_anos'] = df['emp_length'].apply(parse_emp_length)
print('emp_length_anos criado. Valores unicos:', sorted(df['emp_length_anos'].dropna().unique()))

low_card_cols = ['grade', 'sub_grade', 'home_ownership', 'purpose', 'verification_status',
                  'addr_state', 'initial_list_status', 'application_type']
for c in low_card_cols:
    before_card = df[c].nunique(dropna=True)
    df[c] = df[c].str.strip().str.lower()
    after_card = df[c].nunique(dropna=True)
    print(f'{c}: cardinalidade antes={before_card} depois={after_card}')


Datas convertidas: ['issue_d', 'earliest_cr_line']


term convertido para numerico (meses). Valores unicos: [np.float64(36.0), np.float64(60.0)]


emp_length_anos criado. Valores unicos: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]
grade: cardinalidade antes=7 depois=7
sub_grade: cardinalidade antes=35 depois=35
home_ownership: cardinalidade antes=6 depois=6


purpose: cardinalidade antes=14 depois=14
verification_status: cardinalidade antes=3 depois=3
addr_state: cardinalidade antes=51 depois=51
initial_list_status: cardinalidade antes=2 depois=2


application_type: cardinalidade antes=2 depois=2


## 4. Descarte de colunas (com motivo explicito)

Antes de descartar, a lista completa e impressa: coluna -> motivo.

In [6]:
drop_reasons = {}

for c in ['id', 'member_id', 'url']:
    drop_reasons[c] = 'Familia A - identificador'

family_B_original = [
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'pymnt_plan',
]
print(f'Familia B (classificacao original): {len(family_B_original)} colunas')
for c in family_B_original:
    drop_reasons[c] = 'Familia B - pos-originacao (leakage)'
drop_reasons['funded_amnt_inv'] = ('Familia B (REVISAR resolvido para B) - razao funded_amnt_inv/funded_amnt '
                                    'sobe de 0.235 (2007) a 0.9996 (2015): valor se completa apos a originacao')

for c in ['emp_title', 'title', 'desc', 'zip_code']:
    drop_reasons[c] = 'Familia D - texto livre'

zero_var_cols = ['policy_code', 'disbursement_method', 'pymnt_plan', 'hardship_flag', 'out_prncp', 'out_prncp_inv']
for c in zero_var_cols:
    if c in drop_reasons:
        drop_reasons[c] = drop_reasons[c] + ' + variancia zero na populacao'
    else:
        drop_reasons[c] = 'Variancia zero na populacao (cardinalidade = 1)'

block_dec2015 = ['open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
                  'total_bal_il', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
                  'inq_fi', 'total_cu_tl', 'inq_last_12m']
print(f'Bloco nascido dez/2015: {len(block_dec2015)} colunas')
for c in block_dec2015:
    drop_reasons[c] = 'Bloco nascido dez/2015 (~97.8% nulo na populacao, corte de maturidade termina antes do rollout completar)'

null100_cols = ['revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high',
                 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc',
                 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il',
                 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths',
                 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog',
                 'next_pymnt_d']
for c in null100_cols:
    if c in drop_reasons:
        drop_reasons[c] = drop_reasons[c] + ' + 100% nula na populacao'
    else:
        drop_reasons[c] = '100% nula na populacao (nasceu apos o fim da janela da populacao, mar/2017)'

drop_reasons['addr_state'] = ('Rare-category / alta cardinalidade (51 niveis, 1/3 sem amostra suficiente) - '
                                'dropada por decisao registrada em cleaning_decisions.md')

drop_reasons['emp_length'] = 'Transformacao mecanica - substituida por emp_length_anos (parseada)'

print()
print(f'Total de colunas a descartar: {len(drop_reasons)}')
for c, reason in sorted(drop_reasons.items()):
    print(f'  {c}: {reason}')


Familia B (classificacao original): 37 colunas
Bloco nascido dez/2015: 13 colunas

Total de colunas a descartar: 75
  addr_state: Rare-category / alta cardinalidade (51 niveis, 1/3 sem amostra suficiente) - dropada por decisao registrada em cleaning_decisions.md
  all_util: Bloco nascido dez/2015 (~97.8% nulo na populacao, corte de maturidade termina antes do rollout completar)
  collection_recovery_fee: Familia B - pos-originacao (leakage)
  debt_settlement_flag: Familia B - pos-originacao (leakage)
  debt_settlement_flag_date: Familia B - pos-originacao (leakage)
  deferral_term: Familia B - pos-originacao (leakage)
  desc: Familia D - texto livre
  disbursement_method: Variancia zero na populacao (cardinalidade = 1)
  emp_length: Transformacao mecanica - substituida por emp_length_anos (parseada)
  emp_title: Familia D - texto livre
  funded_amnt_inv: Familia B (REVISAR resolvido para B) - razao funded_amnt_inv/funded_amnt sobe de 0.235 (2007) a 0.9996 (2015): valor se completa apos

In [7]:
cols_to_drop = [c for c in drop_reasons if c in df.columns]
missing = [c for c in drop_reasons if c not in df.columns]
if missing:
    print(f'ATENCAO: colunas na lista de descarte que nao existem no dataframe: {missing}')

shape_before = df.shape
df = df.drop(columns=cols_to_drop)
print(f'Shape antes: {shape_before} -> depois: {df.shape}')


Shape antes: (673548, 153) -> depois: (673548, 78)


## 5. Familia E (EVAL_ONLY) e exclusoes provisorias

Familia E nunca entra em nenhum feature set, em nenhuma fase. int_rate, grade e sub_grade
sao mantidas no dataset mas marcadas como exclusao provisoria (ver "Deferred to modeling"
em cleaning_decisions.md) — a serem resolvidas na modelagem, com versao comparativa.

In [8]:
EVAL_ONLY = ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
print('EVAL_ONLY (nunca features, em nenhuma fase):', EVAL_ONLY)
print('Todas presentes no dataframe?', all(c in df.columns for c in EVAL_ONLY))

PROVISIONAL_EXCLUDE = ['int_rate', 'grade', 'sub_grade']
print()
print('PROVISIONAL_EXCLUDE (mantidas no dataset, excluidas do feature set ate a modelagem')
print('comparativa decidir se entram):', PROVISIONAL_EXCLUDE)
print('Todas presentes no dataframe?', all(c in df.columns for c in PROVISIONAL_EXCLUDE))


EVAL_ONLY (nunca features, em nenhuma fase): ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
Todas presentes no dataframe? True

PROVISIONAL_EXCLUDE (mantidas no dataset, excluidas do feature set ate a modelagem
comparativa decidir se entram): ['int_rate', 'grade', 'sub_grade']
Todas presentes no dataframe? True


## 6. MNAR — flag + sentinela 999 (mths_since_*) e -1 (emp_length_anos)

In [9]:
mnar_cols_999 = ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_recent_bc_dlq',
                  'mths_since_recent_revol_delinq', 'mths_since_last_major_derog']

for c in mnar_cols_999:
    pct_null_before = df[c].isnull().mean() * 100
    flag_col = f'{c}_missing'
    df[flag_col] = df[c].isnull().astype(int)
    df[c] = df[c].fillna(999)
    pct_null_after = df[c].isnull().mean() * 100
    print(f'{c}: %nulo antes={pct_null_before:.2f}% depois={pct_null_after:.2f}% | {flag_col} soma={df[flag_col].sum():,}')

print()
pct_null_before = df['emp_length_anos'].isnull().mean() * 100
df['emp_length_missing'] = df['emp_length_anos'].isnull().astype(int)
df['emp_length_anos'] = df['emp_length_anos'].fillna(-1)
pct_null_after = df['emp_length_anos'].isnull().mean() * 100
print(f'emp_length_anos: %nulo antes={pct_null_before:.2f}% depois={pct_null_after:.2f}% | '
      f'emp_length_missing soma={df["emp_length_missing"].sum():,}')


mths_since_last_delinq: %nulo antes=51.77% depois=0.00% | mths_since_last_delinq_missing soma=348,666
mths_since_last_record: %nulo antes=84.58% depois=0.00% | mths_since_last_record_missing soma=569,678
mths_since_recent_bc_dlq: %nulo antes=77.07% depois=0.00% | mths_since_recent_bc_dlq_missing soma=519,104
mths_since_recent_revol_delinq: %nulo antes=67.76% depois=0.00% | mths_since_recent_revol_delinq_missing soma=456,366
mths_since_last_major_derog: %nulo antes=75.42% depois=0.00% | mths_since_last_major_derog_missing soma=508,015

emp_length_anos: %nulo antes=5.58% depois=0.00% | emp_length_missing soma=37,567


## 7. Bloco de bureau pre-2012 — flag era_pre_2012 + sentinela -1

Colunas identificadas programaticamente pela sobreposicao com a mascara
`tot_cur_bal.isna()` (ja validada em 02_cleaning.ipynb), nao por uma lista fixa.

In [10]:
ref_mask = df['tot_cur_bal'].isna()
already_handled = set(EVAL_ONLY) | set(mnar_cols_999) | {'emp_length_anos', 'emp_length_missing', 'target'}
candidate_cols = [c for c in df.columns if c not in already_handled and df[c].isnull().sum() > 0]

bureau_2012_cols = []
for c in candidate_cols:
    col_mask = df[c].isna()
    if col_mask.sum() == 0:
        continue
    overlap_of_ref = (col_mask & ref_mask).sum() / ref_mask.sum() * 100 if ref_mask.sum() > 0 else 0
    overlap_of_col = (col_mask & ref_mask).sum() / col_mask.sum() * 100 if col_mask.sum() > 0 else 0
    if overlap_of_ref > 95 and overlap_of_col > 95:
        bureau_2012_cols.append(c)

print(f'Colunas do bloco bureau 2012 identificadas via mascara de tot_cur_bal: {len(bureau_2012_cols)}')
print(bureau_2012_cols)
print('(nota: o contrato estimava "~24"; a contagem exata medida via sobreposicao de mascara prevalece)')


Colunas do bloco bureau 2012 identificadas via mascara de tot_cur_bal: 21
['tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'avg_cur_bal', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'tot_hi_cred_lim', 'total_il_high_credit_limit']
(nota: o contrato estimava "~24"; a contagem exata medida via sobreposicao de mascara prevalece)


In [11]:
df['era_pre_2012'] = ref_mask.astype(int)
for c in bureau_2012_cols:
    df[c] = df[c].fillna(-1)

print(f'era_pre_2012 - soma da flag: {df["era_pre_2012"].sum():,}')
print(f'%_populacao com era_pre_2012=1: {df["era_pre_2012"].mean() * 100:.4f}%')
print('Compara com a sobreposicao ja medida em 02_cleaning.ipynb (~10.0255% / 67.527 linhas).')


era_pre_2012 - soma da flag: 67,527
%_populacao com era_pre_2012=1: 10.0256%
Compara com a sobreposicao ja medida em 02_cleaning.ipynb (~10.0255% / 67.527 linhas).


## 8. Consolidacao de categorias raras

In [12]:
print('home_ownership - antes:')
print(df['home_ownership'].value_counts())

df['home_ownership'] = df['home_ownership'].replace({'any': 'other', 'none': 'other'})

print()
print('home_ownership - depois:')
print(df['home_ownership'].value_counts())


home_ownership - antes:
home_ownership
mortgage    320795
rent        285754
own          66808
other          144
none            45
any              2
Name: count, dtype: int64

home_ownership - depois:
home_ownership
mortgage    320795
rent        285754
own          66808
other          191
Name: count, dtype: int64


In [13]:
print('purpose - antes:')
print(df['purpose'].value_counts())

df['purpose'] = df['purpose'].replace({'wedding': 'other', 'educational': 'other', 'renewable_energy': 'other'})

print()
print('purpose - depois:')
print(df['purpose'].value_counts())


purpose - antes:
purpose
debt_consolidation    385923
credit_card           159055
home_improvement       39094
other                  36375
major_purchase         14076
small_business          8635
car                     7786
medical                 7245
moving                  4815
vacation                4426
house                   2990
wedding                 2290
renewable_energy         512
educational              326
Name: count, dtype: int64



purpose - depois:
purpose
debt_consolidation    385923
credit_card           159055
other                  39503
home_improvement       39094
major_purchase         14076
small_business          8635
car                     7786
medical                 7245
moving                  4815
vacation                4426
house                   2990
Name: count, dtype: int64


## 9. Nulos remanescentes (sem imputar — so reportar)

In [14]:
remaining_nulls = (df.isnull().mean() * 100)
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f'Colunas com nulo remanescente: {len(remaining_nulls)}')
remaining_nulls.to_frame('%_nulo')


Colunas com nulo remanescente: 23


,%_nulo
dti_joint,99.965407
annual_inc_joint,99.965259
verification_status_joint,99.965259
il_util,98.097389
mths_since_recent_inq,16.971025
mo_sin_old_il_acct,13.294821
num_tl_120dpd_2m,13.134476
num_sats,8.290575
num_bc_sats,8.290575
bc_util,7.989631


## 10. Escrita do dataset analitico

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / 'loans_clean.parquet'
df.to_parquet(parquet_path, index=False)

sample_path = PROCESSED_DIR / 'loans_clean_sample.csv'
df.sample(n=1000, random_state=42).to_csv(sample_path, index=False)

parquet_size_mb = parquet_path.stat().st_size / 1024**2
sample_size_kb = sample_path.stat().st_size / 1024

print(f'Shape final: {df.shape}')
print(f'Parquet: {parquet_path} - {parquet_size_mb:.2f} MB')
print(f'Sample CSV: {sample_path} - {sample_size_kb:.2f} KB')
print(f'Taxa de default final: {df["target"].mean() * 100:.4f}%')


Shape final: (673548, 85)
Parquet: ..\data\processed\loans_clean.parquet - 45.53 MB
Sample CSV: ..\data\processed\loans_clean_sample.csv - 428.12 KB
Taxa de default final: 14.8148%


## 11. Verificacao de integridade final

In [16]:
family_B_all = family_B_original + ['funded_amnt_inv']
survived_B = [c for c in family_B_all if c in df.columns]
print(f'Colunas familia B sobreviventes (esperado 0): {survived_B}')
assert len(survived_B) == 0, 'PARANDO: coluna(s) da familia B sobreviveram ao descarte.'

print()
print('EVAL_ONLY presentes no dataframe:', [c for c in EVAL_ONLY if c in df.columns])
print('EVAL_ONLY faltando:', [c for c in EVAL_ONLY if c not in df.columns])
print('(EVAL_ONLY existe no arquivo para calculo de resultado financeiro e auditoria; a')
print(' exclusao do feature set e responsabilidade da fase de modelagem, nao deste notebook.)')


Colunas familia B sobreviventes (esperado 0): []

EVAL_ONLY presentes no dataframe: ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
EVAL_ONLY faltando: []
(EVAL_ONLY existe no arquivo para calculo de resultado financeiro e auditoria; a
 exclusao do feature set e responsabilidade da fase de modelagem, nao deste notebook.)


In [17]:
id_status = 'id foi removida do dataset (familia A - identificador), nao ha o que checar.' if 'id' not in df.columns else str(df['id'].dtype)
print('id:', id_status)
print()
print('dtypes finais:')
print(df.dtypes.to_string())


id: id foi removida do dataset (familia A - identificador), nao ha o que checar.

dtypes finais:
loan_amnt                                        float64
funded_amnt                                      float64
term                                             float64
int_rate                                         float64
installment                                      float64
grade                                                str
sub_grade                                            str
home_ownership                                       str
annual_inc                                       float64
verification_status                                  str
issue_d                                   datetime64[us]
loan_status                                          str
purpose                                              str
dti                                              float64
delinq_2yrs                                      float64
earliest_cr_line                          dateti

In [18]:
expected_object_cols = {'loan_status', 'grade', 'sub_grade', 'home_ownership', 'verification_status',
                          'purpose', 'initial_list_status', 'application_type', 'verification_status_joint'}
unexpected_object = [c for c in df.columns if df[c].dtype == 'object' and c not in expected_object_cols]
print(f'Colunas object nao esperadas (deveriam ser numericas): {unexpected_object}')
if unexpected_object:
    print('ATENCAO: revisar antes de seguir.')
else:
    print('Nenhuma coluna object inesperada. Todas as colunas categoricas remanescentes sao as esperadas.')


Colunas object nao esperadas (deveriam ser numericas): []
Nenhuma coluna object inesperada. Todas as colunas categoricas remanescentes sao as esperadas.


## Diagnostico dos nulos remanescentes

Somente leitura. Nenhuma imputacao, nenhuma remocao, nenhuma alteracao no parquet. Carrega
`data/processed/loans_clean.parquet` (673.548 linhas) diretamente — nao reexecuta a
construcao do dataset.

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

df = pd.read_parquet(Path('..') / 'data' / 'processed' / 'loans_clean.parquet')
print(f'Shape carregado: {df.shape}')
assert len(df) == 673548, f'N inesperado: {len(df)} (esperado 673548)'

df['ano_emissao'] = df['issue_d'].dt.year
print('OK - parquet carregado, nao sera reescrito.')


Shape carregado: (673548, 85)
OK - parquet carregado, nao sera reescrito.


### Parte 1 - Mecanismo das 10 colunas do cluster de marco/2012

Leitura esperada: se o %nulo cai para ~0% depois de 2012, e rollout de coluna (ausencia
estrutural temporal). Se o %nulo e estavel ao longo de todas as safras, e ausencia
estrutural por evento (o tomador nunca teve o evento).

In [20]:
cluster_mar2012 = ['mort_acc', 'total_bc_limit', 'total_bal_ex_mort', 'acc_open_past_24mths',
                    'bc_util', 'bc_open_to_buy', 'percent_bc_gt_75', 'mths_since_recent_bc',
                    'mths_since_recent_inq', 'mo_sin_old_il_acct']

years = list(range(2007, 2016))
rows = []
for c in cluster_mar2012:
    null_mask = df[c].isnull()
    row = {'coluna': c, '%nulo_populacao': round(null_mask.mean() * 100, 4)}
    for y in years:
        year_mask = df['ano_emissao'] == y
        row[f'%nulo_{y}'] = round(df.loc[year_mask, c].isnull().mean() * 100, 2)
    taxa_nulo = df.loc[null_mask, 'target'].mean() * 100
    taxa_preenchido = df.loc[~null_mask, 'target'].mean() * 100
    row['default_nulo_%'] = round(taxa_nulo, 2)
    row['default_preenchido_%'] = round(taxa_preenchido, 2)
    row['diferenca_pp'] = round(taxa_nulo - taxa_preenchido, 2)
    n_null = int(null_mask.sum())
    overlap = (null_mask & (df['era_pre_2012'] == 1)).sum() / n_null * 100 if n_null > 0 else np.nan
    row['%_overlap_era_pre_2012'] = round(overlap, 2)
    rows.append(row)

cluster_table = pd.DataFrame(rows).set_index('coluna')
cluster_table


,%nulo_populacao,%nulo_2007,%nulo_2008,%nulo_2009,%nulo_2010,%nulo_2011,%nulo_2012,%nulo_2013,%nulo_2014,%nulo_2015,default_nulo_%,default_preenchido_%,diferenca_pp,%_overlap_era_pre_2012
coluna,,,,,,,,,,,,,,
mort_acc,7.0197,100.0,100.0,100.0,100.0,100.0,14.04,0.00,0.00,0.00,14.56,14.83,-0.27,100.00
total_bc_limit,7.0197,100.0,100.0,100.0,100.0,100.0,14.04,0.00,0.00,0.00,14.56,14.83,-0.27,100.00
total_bal_ex_mort,7.0197,100.0,100.0,100.0,100.0,100.0,14.04,0.00,0.00,0.00,14.56,14.83,-0.27,100.00
acc_open_past_24mths,7.0197,100.0,100.0,100.0,100.0,100.0,14.04,0.00,0.00,0.00,14.56,14.83,-0.27,100.00
bc_util,7.9896,100.0,100.0,100.0,100.0,100.0,15.08,0.79,1.16,1.07,14.85,14.81,0.04,88.39
bc_open_to_buy,7.9286,100.0,100.0,100.0,100.0,100.0,15.03,0.75,1.08,1.00,14.83,14.81,0.01,89.04
percent_bc_gt_75,7.9813,100.0,100.0,100.0,100.0,100.0,15.03,0.75,1.14,1.09,14.84,14.81,0.02,88.46
mths_since_recent_bc,7.8612,100.0,100.0,100.0,100.0,100.0,14.89,0.65,0.99,0.96,14.81,14.82,-0.01,89.75
mths_since_recent_inq,16.9710,100.0,100.0,100.0,100.0,100.0,24.90,10.82,9.48,11.04,12.11,15.37,-3.25,43.69


### Parte 2 - Agrupamento por mascara identica

In [21]:
mask_signatures = {}
for c in cluster_mar2012:
    sig = df[c].isnull().values.tobytes()
    mask_signatures.setdefault(sig, []).append(c)

groups = list(mask_signatures.values())
print(f'Grupos de mascara EXATAMENTE identica encontrados: {len(groups)}')
for g in groups:
    pct_null = df[g[0]].isnull().mean() * 100
    print(f'  Grupo (tamanho {len(g)}): {g} | %nulo = {pct_null:.4f}%')


Grupos de mascara EXATAMENTE identica encontrados: 7
  Grupo (tamanho 4): ['mort_acc', 'total_bc_limit', 'total_bal_ex_mort', 'acc_open_past_24mths'] | %nulo = 7.0197%
  Grupo (tamanho 1): ['bc_util'] | %nulo = 7.9896%
  Grupo (tamanho 1): ['bc_open_to_buy'] | %nulo = 7.9286%
  Grupo (tamanho 1): ['percent_bc_gt_75'] | %nulo = 7.9813%
  Grupo (tamanho 1): ['mths_since_recent_bc'] | %nulo = 7.8612%
  Grupo (tamanho 1): ['mths_since_recent_inq'] | %nulo = 16.9710%
  Grupo (tamanho 1): ['mo_sin_old_il_acct'] | %nulo = 13.2948%


In [22]:
masks = {c: df[c].isnull() for c in cluster_mar2012}
overlap_matrix = pd.DataFrame(index=cluster_mar2012, columns=cluster_mar2012, dtype=float)

for c1 in cluster_mar2012:
    for c2 in cluster_mar2012:
        n1 = masks[c1].sum()
        n2 = masks[c2].sum()
        inter = (masks[c1] & masks[c2]).sum()
        denom = min(n1, n2)
        overlap_matrix.loc[c1, c2] = round(inter / denom * 100, 2) if denom > 0 else np.nan

print('Matriz de sobreposicao par a par (% da mascara MENOR contida na mascara da coluna cruzada):')
overlap_matrix


Matriz de sobreposicao par a par (% da mascara MENOR contida na mascara da coluna cruzada):


,mort_acc,total_bc_limit,total_bal_ex_mort,acc_open_past_24mths,bc_util,bc_open_to_buy,percent_bc_gt_75,mths_since_recent_bc,mths_since_recent_inq,mo_sin_old_il_acct
mort_acc,100.0,100.0,100.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00
total_bc_limit,100.0,100.0,100.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00
total_bal_ex_mort,100.0,100.0,100.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00
acc_open_past_24mths,100.0,100.0,100.0,100.0,100.00,100.00,100.00,100.00,100.00,100.00
bc_util,100.0,100.0,100.0,100.0,100.00,100.00,99.34,99.85,89.54,88.62
bc_open_to_buy,100.0,100.0,100.0,100.0,100.00,100.00,100.00,99.85,90.14,89.25
percent_bc_gt_75,100.0,100.0,100.0,100.0,99.34,100.00,100.00,100.00,89.57,88.67
mths_since_recent_bc,100.0,100.0,100.0,100.0,99.85,99.85,100.00,100.00,90.82,89.95
mths_since_recent_inq,100.0,100.0,100.0,100.0,89.54,90.14,89.57,90.82,100.00,60.73
mo_sin_old_il_acct,100.0,100.0,100.0,100.0,88.62,89.25,88.67,89.95,60.73,100.00


### Parte 3 - Nulo esparso (grupo residual)

pub_rec_bankruptcies, revol_util, chargeoff_within_12_mths, collections_12_mths_ex_med,
tax_liens, dti.

In [23]:
sparse_cols = ['pub_rec_bankruptcies', 'revol_util', 'chargeoff_within_12_mths',
               'collections_12_mths_ex_med', 'tax_liens', 'dti']

any_null_mask = df[sparse_cols].isnull().any(axis=1)
n_unique_rows = int(any_null_mask.sum())
print(f'Linhas unicas com nulo em pelo menos uma das 6 colunas: {n_unique_rows}')

taxa_default_subset = df.loc[any_null_mask, 'target'].mean() * 100
taxa_default_pop = df['target'].mean() * 100
print(f'Taxa de default nesse subconjunto: {taxa_default_subset:.4f}%')
print(f'Taxa de default da populacao inteira: {taxa_default_pop:.4f}%')
print(f'Diferenca: {taxa_default_subset - taxa_default_pop:.4f} pp')


Linhas unicas com nulo em pelo menos uma das 6 colunas: 1079
Taxa de default nesse subconjunto: 18.3503%
Taxa de default da populacao inteira: 14.8148%
Diferenca: 3.5355 pp


In [24]:
dist_by_year = df.loc[any_null_mask, 'ano_emissao'].value_counts().sort_index()
dist_by_year_pct = (dist_by_year / dist_by_year.sum() * 100).round(2)
print('Distribuicao dessas linhas por ano de issue_d (contagem e % do subconjunto):')
pd.DataFrame({'contagem': dist_by_year, '%_do_subconjunto': dist_by_year_pct})


Distribuicao dessas linhas por ano de issue_d (contagem e % do subconjunto):


,contagem,%_do_subconjunto
ano_emissao,,
2007,246,22.80
2008,458,42.45
2009,17,1.58
2010,19,1.76
2011,9,0.83
2012,47,4.36
2013,78,7.23
2014,77,7.14
2015,128,11.86


In [25]:
example_cols = ['issue_d', 'term', 'loan_amnt', 'annual_inc', 'emp_length_anos',
                 'era_pre_2012', 'loan_status'] + sparse_cols
examples = df.loc[any_null_mask, example_cols].sample(n=10, random_state=42)
examples


,issue_d,term,loan_amnt,annual_inc,emp_length_anos,era_pre_2012,loan_status,pub_rec_bankruptcies,revol_util,chargeoff_within_12_mths,collections_12_mths_ex_med,tax_liens,dti
484829,2008-02-01,36.0,3975.0,24000.0,2.0,1,Fully Paid,NaN,82.3,0.0,0.0,0.0,21.90
484960,2008-01-01,36.0,4000.0,65000.0,10.0,1,Fully Paid,NaN,5.3,0.0,0.0,0.0,9.25
485149,2007-12-01,36.0,12000.0,105000.0,0.0,1,Fully Paid,NaN,53.8,0.0,0.0,0.0,18.25
485000,2008-01-01,36.0,20000.0,57860.0,1.0,1,Fully Paid,NaN,70.0,0.0,0.0,0.0,10.14
485252,2007-10-01,36.0,7000.0,55400.0,2.0,1,Fully Paid,NaN,33.8,0.0,0.0,0.0,10.05
485031,2008-01-01,36.0,10000.0,55000.0,0.0,1,Fully Paid,NaN,20.4,0.0,0.0,0.0,7.59
542287,2013-09-01,36.0,14000.0,122000.0,10.0,0,Fully Paid,0.0,NaN,0.0,0.0,0.0,7.88
545034,2013-08-01,60.0,11200.0,65000.0,10.0,0,Fully Paid,0.0,NaN,0.0,0.0,0.0,10.13
485373,2007-07-01,36.0,8500.0,18000.0,3.0,1,Fully Paid,NaN,26.9,NaN,NaN,NaN,6.40
558082,2013-07-01,36.0,15000.0,64000.0,9.0,0,Fully Paid,0.0,NaN,0.0,0.0,0.0,15.62


### Parte 4 - Confirmacao do grupo de descarte

In [26]:
il_util_by_year = df.groupby('ano_emissao')['il_util'].apply(lambda s: s.isnull().mean() * 100).round(2)
print('il_util - %nulo por ano de issue_d (populacao, 2007-2015):')
print(il_util_by_year)
print()
print('Bloco dez/2015 ja descartado (ex: open_acc_6m) tinha o mesmo padrao dentro da populacao:')
print('100% nulo ate 2014, caindo para ~94.9% em 2015 (rollout tardio, ver 02_cleaning.ipynb).')


il_util - %nulo por ano de issue_d (populacao, 2007-2015):
ano_emissao
2007    100.00
2008    100.00
2009    100.00
2010    100.00
2011    100.00
2012    100.00
2013    100.00
2014    100.00
2015     95.47
Name: il_util, dtype: float64

Bloco dez/2015 ja descartado (ex: open_acc_6m) tinha o mesmo padrao dentro da populacao:
100% nulo ate 2014, caindo para ~94.9% em 2015 (rollout tardio, ver 02_cleaning.ipynb).


In [27]:
joint_cols = ['dti_joint', 'annual_inc_joint', 'verification_status_joint']
joint_app_mask = df['application_type'] == 'joint app'
print(f'Linhas com application_type == "joint app": {int(joint_app_mask.sum())}')
print()

for c in joint_cols:
    non_null_mask = df[c].notna()
    match = (non_null_mask == joint_app_mask).all()
    n_non_null = int(non_null_mask.sum())
    print(f'{c}: linhas nao-nulas={n_non_null} | mascara identica a application_type==joint app? {match}')


Linhas com application_type == "joint app": 234

dti_joint: linhas nao-nulas=233 | mascara identica a application_type==joint app? False
annual_inc_joint: linhas nao-nulas=234 | mascara identica a application_type==joint app? True
verification_status_joint: linhas nao-nulas=234 | mascara identica a application_type==joint app? True


## Finalizacao - tratamento dos nulos remanescentes

Implementa docs/cleaning_decisions.md ("Remaining nulls - resolved by mechanism"). Se algo
aqui divergir do documento, o notebook para (assert) e sinaliza. Recarrega o parquet do
zero (nao depende do estado da secao de diagnostico acima).

In [28]:
df = pd.read_parquet(Path('..') / 'data' / 'processed' / 'loans_clean.parquet')
print(f'Shape recarregado: {df.shape}')
assert len(df) == 673548, f'N inesperado: {len(df)} (esperado 673548)'


Shape recarregado: (673548, 85)


## 1. Remocao das linhas joint app (dti sistematicamente distorcido)

In [29]:
n_before = len(df)
joint_mask = df['application_type'] == 'joint app'
n_joint = int(joint_mask.sum())
print(f'Linhas joint app encontradas: {n_joint} (esperado: 234)')
assert n_joint == 234, f'PARANDO: esperava 234 linhas joint app, encontrou {n_joint} - diverge do contrato.'

df = df.loc[~joint_mask].reset_index(drop=True)
print(f'N antes: {n_before:,} -> N depois: {len(df):,}')


Linhas joint app encontradas: 234 (esperado: 234)


N antes: 673,548 -> N depois: 673,314


## 2. Descarte de colunas do Mecanismo 4 (janela de coleta)

In [30]:
drop_reasons_2 = {
    'il_util': ('Mecanismo 4 - mesmo rollout tardio (dez/2015) do bloco de 13 colunas ja descartado; '
                '100% nulo ate 2014, 95.47% em 2015. Fora de escopo por descuido, nao por criterio.'),
    'dti_joint': 'Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.',
    'annual_inc_joint': 'Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.',
    'verification_status_joint': 'Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.',
}
print('Colunas a descartar nesta etapa:')
for c, r in drop_reasons_2.items():
    print(f'  {c}: {r}')

shape_before = df.shape
df = df.drop(columns=list(drop_reasons_2.keys()))
print(f'Shape antes: {shape_before} -> depois: {df.shape}')


Colunas a descartar nesta etapa:
  il_util: Mecanismo 4 - mesmo rollout tardio (dez/2015) do bloco de 13 colunas ja descartado; 100% nulo ate 2014, 95.47% em 2015. Fora de escopo por descuido, nao por criterio.
  dti_joint: Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.
  annual_inc_joint: Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.
  verification_status_joint: Mecanismo 4 - so existe para joint app, que acabou de ser removido do dataset.
Shape antes: (673314, 85) -> depois: (673314, 81)


## 3. Mecanismo 1 - sentinela -1, sem flag nova (6 colunas)

mort_acc, total_bc_limit, total_bal_ex_mort, acc_open_past_24mths (rollout mar/2012,
mascara identica entre si) mais num_bc_sats e num_sats (rollout jun/2012, mascara propria,
mas mesmo mecanismo). Confirma 100% de overlap com era_pre_2012=1 antes de preencher cada
uma — o assert e sobre o overlap com a flag, nao sobre igualdade de mascara entre colunas.
Se alguma nao bater 100%, para.

In [31]:
mech1_cols = ['mort_acc', 'total_bc_limit', 'total_bal_ex_mort', 'acc_open_past_24mths',
              'num_bc_sats', 'num_sats']

for c in mech1_cols:
    null_mask = df[c].isnull()
    n_null = int(null_mask.sum())
    overlap = (null_mask & (df['era_pre_2012'] == 1)).sum() / n_null * 100 if n_null > 0 else np.nan
    print(f'{c}: nulos={n_null:,} | overlap com era_pre_2012=1: {overlap:.4f}%')
    assert overlap == 100.0, f'PARANDO: {c} tem overlap {overlap:.4f}% com era_pre_2012, esperado 100% - diverge do contrato.'

for c in mech1_cols:
    df[c] = df[c].fillna(-1)
print()
print('Preenchidas com sentinela -1 (sem flag nova - ja cobertas por era_pre_2012).')


mort_acc: nulos=47,281 | overlap com era_pre_2012=1: 100.0000%
total_bc_limit: nulos=47,281 | overlap com era_pre_2012=1: 100.0000%
total_bal_ex_mort: nulos=47,281 | overlap com era_pre_2012=1: 100.0000%
acc_open_past_24mths: nulos=47,281 | overlap com era_pre_2012=1: 100.0000%
num_bc_sats: nulos=55,841 | overlap com era_pre_2012=1: 100.0000%
num_sats: nulos=55,841 | overlap com era_pre_2012=1: 100.0000%

Preenchidas com sentinela -1 (sem flag nova - ja cobertas por era_pre_2012).


## 4. Mecanismo 2 - sentinela 999 (6 colunas de causa mista) + flag para mths_since_recent_inq

In [32]:
mech2_cols = ['bc_util', 'bc_open_to_buy', 'percent_bc_gt_75', 'mths_since_recent_bc',
              'mo_sin_old_il_acct', 'mths_since_recent_inq']

df['mths_since_recent_inq_missing'] = df['mths_since_recent_inq'].isnull().astype(int)

for c in mech2_cols:
    pct_null_before = df[c].isnull().mean() * 100
    n_null_before = int(df[c].isnull().sum())
    df[c] = df[c].fillna(999)
    print(f'{c}: %nulo antes={pct_null_before:.4f}% | linhas preenchidas com 999={n_null_before:,}')

print()
print(f'Flag mths_since_recent_inq_missing - soma: {df["mths_since_recent_inq_missing"].sum():,}')


bc_util: %nulo antes=7.9918% | linhas preenchidas com 999=53,810
bc_open_to_buy: %nulo antes=7.9308% | linhas preenchidas com 999=53,399
percent_bc_gt_75: %nulo antes=7.9835% | linhas preenchidas com 999=53,754
mths_since_recent_bc: %nulo antes=7.8635% | linhas preenchidas com 999=52,946
mo_sin_old_il_acct: %nulo antes=13.2990% | linhas preenchidas com 999=89,544
mths_since_recent_inq: %nulo antes=16.9741% | linhas preenchidas com 999=114,289

Flag mths_since_recent_inq_missing - soma: 114,289


## 4b. num_tl_120dpd_2m - sentinela -1 + flag propria

Mecanismo 2, mas sinal invertido de mths_since_recent_inq: aqui a ausencia e falta de
informacao recente do bureau (correlaciona com risco), nao ausencia de evento (que
correlaciona com seguranca). E uma contagem, nao um "meses desde"; nao ha ordem
"maior=mais seguro" a preservar, entao a sentinela e -1 (nao 999).

In [33]:
pct_null_before = df['num_tl_120dpd_2m'].isnull().mean() * 100
n_null_before = int(df['num_tl_120dpd_2m'].isnull().sum())

df['num_tl_120dpd_2m_missing'] = df['num_tl_120dpd_2m'].isnull().astype(int)
default_rate_flag = df.loc[df['num_tl_120dpd_2m_missing'] == 1, 'target'].mean() * 100

df['num_tl_120dpd_2m'] = df['num_tl_120dpd_2m'].fillna(-1)

print(f'num_tl_120dpd_2m: %nulo antes={pct_null_before:.4f}% | linhas preenchidas com -1={n_null_before:,}')
print(f'Flag num_tl_120dpd_2m_missing - soma: {df["num_tl_120dpd_2m_missing"].sum():,}')
print(f'Taxa de default dentro da flag: {default_rate_flag:.4f}%')


num_tl_120dpd_2m: %nulo antes=13.1390% | linhas preenchidas com -1=88,467
Flag num_tl_120dpd_2m_missing - soma: 88,467
Taxa de default dentro da flag: 15.8172%


## 5. Mecanismo 3 - flag agregada + mediana (6 colunas de nulo esparso)

In [34]:
sparse_cols = ['pub_rec_bankruptcies', 'revol_util', 'chargeoff_within_12_mths',
               'collections_12_mths_ex_med', 'tax_liens', 'dti']

any_null_mask = df[sparse_cols].isnull().any(axis=1)
n_sparse = int(any_null_mask.sum())
df['sparse_bureau_missing'] = any_null_mask.astype(int)
print(f'sparse_bureau_missing - soma: {n_sparse:,} (esperado proximo de 1.079, menor pela remocao dos joint apps)')

default_rate_sparse = df.loc[any_null_mask, 'target'].mean() * 100
print(f'Taxa de default dentro da flag sparse_bureau_missing: {default_rate_sparse:.4f}%')
print()

for c in sparse_cols:
    median_val = df[c].median()
    n_null = int(df[c].isnull().sum())
    df[c] = df[c].fillna(median_val)
    print(f'{c}: mediana usada={median_val} | linhas imputadas={n_null:,}')


sparse_bureau_missing - soma: 1,077 (esperado proximo de 1.079, menor pela remocao dos joint apps)
Taxa de default dentro da flag sparse_bureau_missing: 18.2916%

pub_rec_bankruptcies: mediana usada=0.0 | linhas imputadas=697
revol_util: mediana usada=55.0 | linhas imputadas=378
chargeoff_within_12_mths: mediana usada=0.0 | linhas imputadas=56
collections_12_mths_ex_med: mediana usada=0.0 | linhas imputadas=56
tax_liens: mediana usada=0.0 | linhas imputadas=39
dti: mediana usada=17.06 | linhas imputadas=0


## 6. Verificacao final de integridade

In [35]:
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(f'Colunas com nulo remanescente: {len(remaining_nulls)}')
if len(remaining_nulls) > 0:
    print(remaining_nulls)
assert len(remaining_nulls) == 0, 'PARANDO: ainda ha nulos no dataset.'
print('Nenhum nulo remanescente. OK.')


Colunas com nulo remanescente: 0
Nenhum nulo remanescente. OK.


In [36]:
family_B_all = family_B_original + ['funded_amnt_inv']
survived_B = [c for c in family_B_all if c in df.columns]
print(f'Colunas familia B sobreviventes (esperado 0): {survived_B}')
assert len(survived_B) == 0, 'PARANDO: coluna(s) da familia B sobreviveram.'

print()
eval_only_present = [c for c in EVAL_ONLY if c in df.columns]
print('EVAL_ONLY presentes:', eval_only_present)
assert eval_only_present == EVAL_ONLY, 'PARANDO: EVAL_ONLY incompleto.'

print()
provisional_present = [c for c in PROVISIONAL_EXCLUDE if c in df.columns]
print('PROVISIONAL_EXCLUDE presentes:', provisional_present)
assert provisional_present == PROVISIONAL_EXCLUDE, 'PARANDO: PROVISIONAL_EXCLUDE incompleto.'


Colunas familia B sobreviventes (esperado 0): []

EVAL_ONLY presentes: ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']

PROVISIONAL_EXCLUDE presentes: ['int_rate', 'grade', 'sub_grade']


In [37]:
expected_object_cols = {'loan_status', 'grade', 'sub_grade', 'home_ownership', 'verification_status',
                          'purpose', 'initial_list_status', 'application_type'}
unexpected_object = [c for c in df.columns if df[c].dtype == 'object' and c not in expected_object_cols]
print(f'Colunas object nao esperadas: {unexpected_object}')
if unexpected_object:
    print('ATENCAO: revisar antes de seguir.')
else:
    print('Nenhuma coluna object inesperada.')

print()
print('dtypes finais:')
print(df.dtypes.to_string())


Colunas object nao esperadas: []
Nenhuma coluna object inesperada.

dtypes finais:
loan_amnt                                        float64
funded_amnt                                      float64
term                                             float64
int_rate                                         float64
installment                                      float64
grade                                                str
sub_grade                                            str
home_ownership                                       str
annual_inc                                       float64
verification_status                                  str
issue_d                                   datetime64[us]
loan_status                                          str
purpose                                              str
dti                                              float64
delinq_2yrs                                      float64
earliest_cr_line                          datetime64[us]
fico_

In [38]:
engineered_flags = ['target', 'era_pre_2012',
                     'mths_since_last_delinq_missing', 'mths_since_last_record_missing',
                     'mths_since_recent_bc_dlq_missing', 'mths_since_recent_revol_delinq_missing',
                     'mths_since_last_major_derog_missing', 'emp_length_missing',
                     'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']

groups = {
    'EVAL_ONLY': [c for c in EVAL_ONLY if c in df.columns],
    'PROVISIONAL_EXCLUDE': [c for c in PROVISIONAL_EXCLUDE if c in df.columns],
    'Engenharia (target/flags/sentinelas)': [c for c in engineered_flags if c in df.columns],
}
already_grouped = set(sum(groups.values(), []))
groups['Feature candidates (familia C)'] = [c for c in df.columns if c not in already_grouped]

for g, cols in groups.items():
    print(f'\n{g} ({len(cols)}):')
    print(cols)

print()
print(f'Shape final: {df.shape}')
print(f'Taxa de default final: {df["target"].mean() * 100:.4f}%')



EVAL_ONLY (5):
['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']

PROVISIONAL_EXCLUDE (3):
['int_rate', 'grade', 'sub_grade']

Engenharia (target/flags/sentinelas) (11):
['target', 'era_pre_2012', 'mths_since_last_delinq_missing', 'mths_since_last_record_missing', 'mths_since_recent_bc_dlq_missing', 'mths_since_recent_revol_delinq_missing', 'mths_since_last_major_derog_missing', 'emp_length_missing', 'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']

Feature candidates (familia C) (65):
['funded_amnt', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cu

## 7. Sobrescrita do dataset analitico

In [39]:
parquet_path = PROCESSED_DIR / 'loans_clean.parquet'
df.to_parquet(parquet_path, index=False)

sample_path = PROCESSED_DIR / 'loans_clean_sample.csv'
df.sample(n=1000, random_state=42).to_csv(sample_path, index=False)

print(f'Parquet: {parquet_path} - {parquet_path.stat().st_size / 1024**2:.2f} MB')
print(f'Sample CSV: {sample_path} - {sample_path.stat().st_size / 1024:.2f} KB')


Parquet: ..\data\processed\loans_clean.parquet - 45.52 MB
Sample CSV: ..\data\processed\loans_clean_sample.csv - 435.02 KB
